In [ ]:
# 구글 드라이브 공유 링크로 파일 다운로드
!pip install -U gdown
!pip install -qqq datasets # huggingface's lib.
!pip install -qqq transformers==4.48.3
!pip install -qqq accelerate==0.28.0
!pip install tensorboard
!pip install -U accelerate

In [ ]:
import gdown
import zipfile
import os

# 파일 ID 입력
file_id = "14Yc5eDlrEvx4wWp6gtSiJ52ne_noEl9u"

# 다운로드 받을 파일 이름 지정
output = "train.zip"

# 다운로드 수행
gdown.download(f"https://drive.google.com/uc?id={file_id}", output, quiet=True)

# 결과 파일 저장 경로
os.makedirs("release", exist_ok=True)

# 압축 해제할 경로
extract_dir = "data"
os.makedirs(extract_dir, exist_ok=True)

# 압축 풀기
with zipfile.ZipFile("train.zip", 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print("압축 해제 완료:", extract_dir)

In [ ]:
# Task 3 에서 검색할 Top-k 유사 이미지 개수
# Task 3 진행시만 사용
TOP_K = 5

In [ ]:
# 추가 패키지 설치 시 '버전 정보' 꼭 명시하여 설치:
# !pip install name_of_package==X.X.X

# 버전 명시 안해주시면 TA가 테스트할 때 버전 충돌이 자주 발생합니다.
# 불이익 받지 않도록 버전 잘 명시해주시기 바랍니다.

In [ ]:
# ===== 설정 & 모델 다운로드 =====
STUDENT_ID = "202502204"
STARGAN_FILE_ID = "1Xx1HGiQz2OFMI53OaiCXbxyBcyiX2dT4"   # stargan_G.pt (iter 130k)
NUM_STYLE = 3
BATCH_SIZE = 32

# 입력 스타일은 test_labels.csv 에서 읽으므로 Task1 모델은 불필요.
if not os.path.exists("stargan_G.pt"):
    gdown.download(f"https://drive.google.com/uc?id={STARGAN_FILE_ID}", "stargan_G.pt", quiet=False)
print("ready: stargan_G.pt")

In [ ]:
# --- StarGAN Generator (학습 코드와 동일 구조) ---
import torch, torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(dim, dim, 3, 1, 1, bias=False),
            nn.InstanceNorm2d(dim, affine=True, track_running_stats=True), nn.ReLU(True),
            nn.Conv2d(dim, dim, 3, 1, 1, bias=False),
            nn.InstanceNorm2d(dim, affine=True, track_running_stats=True))
    def forward(self, x):
        return x + self.block(x)

class Generator(nn.Module):
    def __init__(self, conv_dim=64, c_dim=NUM_STYLE, n_res=6):
        super().__init__()
        layers = [nn.Conv2d(3 + c_dim, conv_dim, 7, 1, 3, bias=False),
                  nn.InstanceNorm2d(conv_dim, affine=True, track_running_stats=True), nn.ReLU(True)]
        cur = conv_dim
        for _ in range(2):
            layers += [nn.Conv2d(cur, cur * 2, 4, 2, 1, bias=False),
                       nn.InstanceNorm2d(cur * 2, affine=True, track_running_stats=True), nn.ReLU(True)]
            cur *= 2
        for _ in range(n_res):
            layers.append(ResidualBlock(cur))
        for _ in range(2):
            layers += [nn.ConvTranspose2d(cur, cur // 2, 4, 2, 1, bias=False),
                       nn.InstanceNorm2d(cur // 2, affine=True, track_running_stats=True), nn.ReLU(True)]
            cur //= 2
        layers += [nn.Conv2d(cur, 3, 7, 1, 3, bias=False), nn.Tanh()]
        self.main = nn.Sequential(*layers)
    def forward(self, x, c):
        c = c.view(c.size(0), c.size(1), 1, 1).expand(-1, -1, x.size(2), x.size(3))
        return self.main(torch.cat([x, c], dim=1))

def label2onehot(labels, dim=NUM_STYLE):
    out = torch.zeros(labels.size(0), dim)
    out[torch.arange(labels.size(0)), labels.long()] = 1.0
    return out

device = "cuda" if torch.cuda.is_available() else "cpu"
gck = torch.load("stargan_G.pt", map_location=device)
G = Generator(c_dim=gck.get("c_dim", NUM_STYLE)).to(device)
G.load_state_dict(gck["model"] if "model" in gck else gck)
G.eval()

# StarGAN 관례: 추론 시 per-instance 정규화 통계를 사용한다(학습이 이 모드로 진행됨).
# eval 기본값인 '누적 running 통계'를 쓰면 전역 색조(pink/red) cast가 생기므로,
# InstanceNorm 의 running 통계를 끄고 입력별 통계를 쓰게 한다. (state_dict 로드 이후에 적용)
for _m in G.modules():
    if isinstance(_m, nn.InstanceNorm2d):
        _m.track_running_stats = False
        _m.running_mean = None
        _m.running_var = None

G_SIZE = gck.get("img_size", 128)
print("G loaded | img_size", G_SIZE)

In [ ]:
# --- 입력(원본) 스타일을 test_labels.csv 에서 읽기 (Task1 분류기 사용 안 함) ---
import csv
from glob import glob

def find_label_csv(base="data"):
    hits = glob(os.path.join(base, "**", "*.csv"), recursive=True)
    for h in hits:                       # 'test' 우선 (자가검증 땐 train_labels.csv)
        if "test" in os.path.basename(h).lower():
            return h
    return hits[0] if hits else None

CSV_PATH = find_label_csv("data")
assert CSV_PATH, "labels CSV(test_labels.csv)를 data/ 에서 찾지 못했습니다."
style_of = {}
with open(CSV_PATH, newline="") as f:
    for row in csv.DictReader(f):
        style_of[row["file_name"].strip()] = int(row["style"])
print("labels csv:", CSV_PATH, "| entries:", len(style_of))

In [ ]:
# --- test 이미지 수집 (data/ 하위 자동 탐색, test 우선) + 생성 전처리 ---
from glob import glob
from PIL import Image
from torchvision import transforms
EXTS = (".jpg", ".jpeg", ".png")

def find_image_dir(base="data"):
    cands = [r for r, _, fs in os.walk(base)
             if any(f.lower().endswith(EXTS) for f in fs)]
    for c in cands:
        if "test" in c.replace("\\", "/").lower():
            return c
    return cands[0] if cands else base

IMG_DIR = find_image_dir("data")
files = [p for p in glob(os.path.join(IMG_DIR, "*")) if p.lower().endswith(EXTS)]
def id_key(p):
    stem = os.path.splitext(os.path.basename(p))[0]
    return (0, int(stem)) if stem.isdigit() else (1, stem)
files = sorted(set(files), key=id_key)
print("image dir:", IMG_DIR, "| images:", len(files))

gen_tf = transforms.Compose([
    transforms.Resize((G_SIZE, G_SIZE)), transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])  # -> [-1,1]

In [ ]:
# --- 변환 생성 & 저장 (앞 20장 0~19.jpg만, 원본 style 제외 2개씩 = 40장) ---
from torchvision.transforms.functional import to_pil_image
out_dir = f"release/{STUDENT_ID}.test.task2"
os.makedirs(out_dir, exist_ok=True)

# 테스트 데이터 앞 20장(0.jpg ~ 19.jpg)만 생성 (TA 안내)
gen_files = [p for p in files
             if os.path.splitext(os.path.basename(p))[0].isdigit()
             and int(os.path.splitext(os.path.basename(p))[0]) < 20]
print("generate for", len(gen_files), "images (0.jpg~19.jpg)")

@torch.no_grad()
def generate(batch_paths):
    pil = [Image.open(p).convert("RGB") for p in batch_paths]
    x = torch.stack([gen_tf(im) for im in pil]).to(device)
    src = [style_of[os.path.basename(p)] for p in batch_paths]   # CSV의 원본 style
    for tgt in range(NUM_STYLE):
        c = label2onehot(torch.full((x.size(0),), tgt)).to(device)
        y = (G(x, c) + 1) / 2          # [-1,1] -> [0,1]
        for k, p in enumerate(batch_paths):
            if src[k] == tgt:
                continue               # 원본 스타일 제외
            num = os.path.splitext(os.path.basename(p))[0]
            # JPEG 고품질 저장(압축 손실 최소화) — 제출 형식 .jpg 유지
            to_pil_image(y[k].clamp(0, 1).cpu()).save(
                os.path.join(out_dir, f"{num}_generate_{tgt}.jpg"),
                quality=95, subsampling=0)

for i in range(0, len(gen_files), BATCH_SIZE):
    generate(gen_files[i:i + BATCH_SIZE])
print("generated:", len(os.listdir(out_dir)), "files (expected 40)")

In [ ]:
import os
fs = sorted(os.listdir(out_dir))
print("task2 outputs:", len(fs), "files (expected 40)")
print(fs[:8])